In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
import pandas as pd

In [3]:
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

driver.maximize_window()

driver.get("https://perchance.org/cx99p75qfg")

WebDriverWait(driver,20).until(
    EC.frame_to_be_available_and_switch_to_it(
        (By.ID,"outputIframeEl")
    )
)

all_data = []

In [ ]:
for search in range(20):

    print(f"\n===== Search {search+1} =====")

    driver.switch_to.default_content()

    WebDriverWait(driver,20).until(
        EC.frame_to_be_available_and_switch_to_it((By.ID,"outputIframeEl"))
    )

    randomize = WebDriverWait(driver,20).until(
        EC.element_to_be_clickable((By.XPATH,"//button[contains(.,'randomize')]"))
    )
    randomize.click()

    soup = BeautifulSoup(driver.page_source,"html.parser")

    youtube_link = None

    for a in soup.find_all("a"):
        if a.text.strip().lower() == "click me":
            youtube_link = a.get("href")
            break

    if not youtube_link:
        print("Click Me link not found")
        continue

    driver.switch_to.new_window("tab")
    driver.get(youtube_link)

    WebDriverWait(driver,20).until(
        EC.presence_of_element_located((By.ID,"contents"))
    )

    videos = []

    last_height = driver.execute_script(
        "return document.documentElement.scrollHeight"
    )

    while len(videos) < 30:

        soup = BeautifulSoup(driver.page_source,"html.parser")

        for video in soup.find_all("a",id="video-title"):

            href = video.get("href")

            if href:

                link = "https://www.youtube.com" + href

                if link not in videos:
                    videos.append(link)

        if len(videos) >= 30:
            break

        driver.execute_script(
            "window.scrollTo(0, document.documentElement.scrollHeight);"
        )

        try:
            WebDriverWait(driver,10).until(
                lambda d: d.execute_script(
                    "return document.documentElement.scrollHeight"
                ) > last_height
            )

            last_height = driver.execute_script(
                "return document.documentElement.scrollHeight"
            )

        except:
            break

    print("Videos:", len(videos))

    for i, link in enumerate(videos[:30], start=1):
        all_data.append({
            "Search": search + 1,
            "Video": i,
            "Link": link
        })

    driver.close()
    driver.switch_to.window(driver.window_handles[0])


===== Search 1 =====
Videos: 33

===== Search 2 =====
Videos: 32

===== Search 3 =====
Videos: 19

===== Search 4 =====
Videos: 32

===== Search 5 =====
Videos: 33

===== Search 6 =====
Videos: 31

===== Search 7 =====
Videos: 40

===== Search 8 =====
Videos: 33

===== Search 9 =====
Videos: 32

===== Search 10 =====
Videos: 36

===== Search 11 =====
Videos: 30

===== Search 12 =====
Videos: 36

===== Search 13 =====
Videos: 31

===== Search 14 =====
Videos: 34

===== Search 15 =====
Videos: 37

===== Search 16 =====
Videos: 37

===== Search 17 =====
Videos: 30

===== Search 18 =====
Videos: 47

===== Search 19 =====
Videos: 32

===== Search 20 =====
Videos: 30


In [8]:
df = pd.DataFrame(all_data)

df.to_csv(
    "youtube_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("-" * 40)
print("Total Searches :", df["Search"].nunique())
print("Total Videos   :", len(df))
print("CSV Saved Successfully")
print("-" * 40)

driver.quit()

----------------------------------------
Total Searches : 20
Total Videos   : 589
CSV Saved Successfully
----------------------------------------
